# Full-Text Search

Search the **text** of filings (2001 onward) with the EDGAR full-text search API at `efts.sec.gov`, then turn the hits into filing URLs.

In [ ]:
Pollis.packages("EDGAR" => "https://github.com/Trumpingtons/EDGAR.jl")

In [ ]:
# import Pkg; Pkg.add(url = "https://github.com/Trumpingtons/EDGAR.jl")
using EDGAR

# Set your contact once via "Set contact" on the webview; it is injected as
# ENV["SEC_USER_AGENT"], which EDGAR.jl reads automatically. Or set it explicitly:
# set_user_agent("Your Name your.email@example.com")

## Run a Search

`full_text_search` URL-encodes the query for you. Restrict by `forms` and a `startdate`/`enddate` range.

In [ ]:
res = full_text_search("climate risk"; forms = "10-K", startdate = "2023-01-01", enddate = "2023-12-31")

println("Total matches: ", res.hits.total.value)

## Read the Hits

Each hit's `_source` carries the filer names, form, and filing date; `_id` is `<accession>:<document>`.

In [ ]:
for h in first(res.hits.hits, 5)
    s = h._source
    println(s.file_date, "  ", rpad(s.form, 8), first(s.display_names))
end

## Build Filing URLs from Hits

Split `_id` into the accession number and document, and combine with the filer's CIK to form the public document URL.

In [ ]:
for h in first(res.hits.hits, 5)
    acc_dashed, doc = split(h._id, ":")
    acc = replace(acc_dashed, "-" => "")
    cik = parse(Int, first(h._source.ciks))      # archive URLs use the un-padded CIK
    println("https://www.sec.gov/Archives/edgar/data/$cik/$acc/$doc")
end

## Summary
- `efts.sec.gov/LATEST/search-index?q=...` searches filing text from 2001 onward; filter with `forms`, `startdt`, `enddt`, `ciks`.
- Each hit's `_source` has `display_names`, `form`, `file_date`; `_id` is `<accession>:<document>`.
- Combine the accession, document, and CIK to build the public filing URL.